# DJO Validation Core (prototype)

Backend prototype for validating a **Declaração Juramentada de Origem (DJO)**
against a Mercosur trade agreement (currently: **ACE 18**).

Responsibilities are kept deliberately separated, matching the eventual
backend package layout:

```
PDF -> DJO extraction (extraction_djo.ipynb) -> json_djo
NCM -> ACE 18 cached dataset (ace18_service.py) -> mercosul_rule
json_djo + mercosul_rule -> Validation Core (this notebook) -> validation_result
```

This notebook does **not** re-implement DJO OCR/extraction or ACE 18
OCR/extraction — it *consumes* `json_djo` (produced by
`extraction_djo.ipynb`) and calls into `ace18_service.py` (the cached,
already-Docling-processed ACE 18 lookup). It also does not implement the
future business "decision flow" — only the 8 validation rules below, each
returning a structured, machine-readable result.

**Principle:** deterministic business rules stay in plain Python (Rules 1,
2, 3, 4, 6, 7, 8). Gemini is used only where genuine semantic language
understanding is required — Rule 5 (matching free-text process inputs
against material-table descriptions) and the optional final human-readable
summary. Gemini never decides PASS/FAIL for the other rules.


## 1-2. Configuration & imports

In [1]:
import sys
import os
import json
from enum import Enum
from typing import Any, Dict, List, Optional, Callable

# --- Make the sibling service modules importable -----------------------
# In the future backend these become real package imports
# (e.g. `from backend.services import ace18_service`); for the notebook
# prototype we just add this folder to sys.path.
CODIGO_DIR = r"C:\Users\juan.s\Documents\Cerdificados_de_origem\codigo"
if CODIGO_DIR not in sys.path:
    sys.path.insert(0, CODIGO_DIR)

from dotenv import load_dotenv

# NOTE: load_dotenv() with no arguments searches upward from the current
# working directory, which in a Jupyter kernel is often NOT this notebook's
# folder (and there is no reliable `__file__` to anchor the search from
# inside a cell). That made it silently fail to find the project-root
# .env in practice. Pointing at the .env explicitly removes that ambiguity.
# `override=True` also makes sure a value you just edited into .env always
# wins over whatever was loaded into this kernel's environment earlier.
PROJECT_ROOT = r"C:\Users\juan.s\Documents\Cerdificados_de_origem"
ENV_PATH = os.path.join(PROJECT_ROOT, ".env")
load_dotenv(dotenv_path=ENV_PATH, override=True)

if not os.environ.get("GEMINI_API_KEY"):
    print(f"WARNING: GEMINI_API_KEY not found after loading {ENV_PATH}. "
          f"Rule 5 and the summary step will report MANUAL_VERIFICATION / be unavailable.")

import ace18_service
import gemini_service


In [2]:
# --- Paths --------------------------------------------------------------
# ACE 18 source PDF + JSON cache built by ace18_service (same files already
# produced by estraction_ac18.ipynb — the cache is reused as-is, so no
# Docling run happens here as long as the cache file already exists).
ACE18_PDF_PATH = r"C:\Users\juan.s\Documents\Cerdificados_de_origem\dados\ACE_018_221_pt (1).pdf"
ACE18_CACHE_PATH = r"C:\Users\juan.s\Documents\Cerdificados_de_origem\dados\reos_ace18_cache.json"

# Tolerance (percentage points) used when comparing a table's calculated
# '% s/Valor FOB' sum against its declared 'Somatório' (Rule 7). Absorbs
# rounding drift across several line items without masking real mismatches.
PERCENTAGE_TOLERANCE = 0.5

# The literal placeholder the DJO extraction pipeline preserves verbatim
# inside material-table cells (see extraction_djo.ipynb). Top-level scalar
# fields of json_djo already collapse this placeholder to "" during
# extraction, but table cells do not — validators must check both forms
# explicitly (see `is_missing_or_not_informado` below).
NAO_INFORMADO = "[Não Informado]"


## 3. Shared models / helpers

Lightweight, dependency-free building blocks used by every rule below.
`ValidationStatus` and the `{rule, status, message, ...}` shape produced by
`make_result` correspond to `models/validation_result.py` in the future
backend layout.


In [3]:
class ValidationStatus(str, Enum):
    PASS = "PASS"
    FAIL = "FAIL"
    MANUAL_VERIFICATION = "MANUAL_VERIFICATION"
    NOT_APPLICABLE = "NOT_APPLICABLE"


def make_result(rule: str, status: ValidationStatus, message: str, **extra) -> Dict[str, Any]:
    """Builds one entry of validation_result["validations"]. Always stores
    the plain string value of `status` so the whole result stays directly
    json.dumps-able without a custom encoder."""
    result: Dict[str, Any] = {"rule": rule, "status": status.value, "message": message}
    result.update(extra)
    return result


def is_blank(value: Any) -> bool:
    """True for None or an empty/whitespace-only string. Does NOT treat the
    literal "[Não Informado]" placeholder as blank by itself — see
    `is_missing_or_not_informado` for the table-cell case where that
    placeholder is preserved verbatim instead of being collapsed to ""."""
    return value is None or (isinstance(value, str) and value.strip() == "")


def is_missing_or_not_informado(value: Any) -> bool:
    """Used for material-table cell validation (Rule 4 and Rule 6), where
    the extraction pipeline preserves "[Não Informado]" as a literal string
    instead of collapsing it to ""."""
    return is_blank(value) or (isinstance(value, str) and value.strip() == NAO_INFORMADO)


def parse_percentage(value: Optional[str]) -> Optional[float]:
    """Parses a Brazilian-formatted percentage string ('13,13%', '0%',
    '100%') into a float. Returns None (never 0.0) when the value is
    blank, '[Não Informado]', or otherwise unparsable — callers must treat
    None as "cannot validate this value", never silently as zero (this
    matters directly for Rule 8's "missing values must be reported as
    missing, not as zero")."""
    if is_missing_or_not_informado(value):
        return None
    cleaned = str(value).strip().replace("%", "").replace(",", ".")
    try:
        return float(cleaned)
    except ValueError:
        return None


# The 4 material tables that can appear under json_djo["Materiales"].
MATERIAL_TABLE_NAMES = [
    "originarios_estado_parte_produtor",
    "originarios_outros_estados_partes",
    "nao_originarios",
    "terceiros_paises_ptc",
]


## Rule 1 — DJO approval status

Informational finding, not a pass/fail judgement on the DJO's validity: a
DJO that has not been approved yet is the normal state for a first
submission, so the "not approved" case is reported as `NOT_APPLICABLE`
rather than `FAIL`.


In [4]:
def validate_djo_approval(json_djo: Dict[str, Any]) -> Dict[str, Any]:
    codigo = json_djo.get("Código da aprovação DJO")
    data_apresentacao = json_djo.get("Data de apresentação")

    if not is_blank(codigo) and not is_blank(data_apresentacao):
        return make_result(
            "djo_approval",
            ValidationStatus.PASS,
            "The DJO has already been approved.",
            details={"codigo_aprovacao": codigo, "data_apresentacao": data_apresentacao},
        )

    return make_result(
        "djo_approval",
        ValidationStatus.NOT_APPLICABLE,
        "The DJO does not have both an approval code and a presentation date yet; it has not been approved.",
    )


## Rule 2 — Producer information

**Ambiguity flagged and resolved explicitly:** `json_djo["Razão social do
produtor"]` and `json_djo["Domicílio legal e parque industrial do
produtor"]` are nested dicts (the DJO form itself splits them into
sub-fields — see `extraction_djo.ipynb`), not plain strings. The rule as
specified just says these two fields must have "non-empty values". We
resolve this by checking the sub-field that carries each field's essence:
`"Razão social"` (the producer's legal name) and `"Endereço"` (the street
address) — `CNPJ/CPF`, `Inscrição Estadual`, `Tel` and `E-mail` are treated
as secondary sub-fields the source DJO form does not always fill in, and
their absence alone does not fail this rule. This is a documented decision,
not a silent guess — revisit here if the intended reading was different
(e.g. requiring every sub-field).


In [5]:
def validate_producer_information(json_djo: Dict[str, Any]) -> Dict[str, Any]:
    razao_social = (json_djo.get("Razão social do produtor") or {}).get("Razão social")
    endereco = (json_djo.get("Domicílio legal e parque industrial do produtor") or {}).get("Endereço")

    missing = []
    if is_blank(razao_social):
        missing.append("Razão social do produtor")
    if is_blank(endereco):
        missing.append("Domicílio legal e parque industrial do produtor")

    if missing:
        return make_result(
            "producer_information",
            ValidationStatus.FAIL,
            "Missing required producer information: " + ", ".join(missing) + ".",
            details={"missing_fields": missing},
        )

    return make_result(
        "producer_information",
        ValidationStatus.PASS,
        "Producer identity and domicile are present.",
    )


## Rule 3 — NCM / ACE 18

Queries `json_djo["Código NCM"]` against the cached ACE 18 table via
`ace18_service.ACE18Service` (reused as-is from `estraction_ac18.ipynb`'s
logic — no OCR/Docling call happens here as long as the JSON cache already
exists on disk). The `AGREEMENT_SERVICES` registry is the extension point
for future agreements: adding one means writing a new service with a
`.query_ncm(...)`-shaped method and registering it here — none of the
other rules need to change.


In [6]:
# Registry of trade-agreement services, keyed by a normalized agreement
# name. FUTURE: add more entries here (e.g. "ACE_XX": build_some_other_service)
# to support additional agreements without touching any validation rule.
_ace18_service_singleton: Optional["ace18_service.ACE18Service"] = None


def _get_ace18_service() -> "ace18_service.ACE18Service":
    global _ace18_service_singleton
    if _ace18_service_singleton is None:
        _ace18_service_singleton = ace18_service.ACE18Service(ACE18_PDF_PATH, ACE18_CACHE_PATH)
    return _ace18_service_singleton


AGREEMENT_SERVICES: Dict[str, Callable[[], "ace18_service.ACE18Service"]] = {
    "ACE_18": _get_ace18_service,
}

# Friendly aliases a frontend might send for the same agreement.
AGREEMENT_ALIASES = {
    "ACE_18": "ACE_18",
    "ACE18": "ACE_18",
    "ACE 18": "ACE_18",
    "MERCOSUL": "ACE_18",
    "MERCOSUR": "ACE_18",
}


def normalize_agreement(agreement: str) -> str:
    key = AGREEMENT_ALIASES.get(str(agreement).strip().upper())
    if key is None:
        raise ValueError(
            f"Unsupported agreement '{agreement}'. Supported: {sorted(set(AGREEMENT_ALIASES.values()))}"
        )
    return key


def validate_ncm_ace18(json_djo: Dict[str, Any], agreement: str) -> Dict[str, Any]:
    ncm = json_djo.get("Código NCM")
    if is_blank(ncm):
        return make_result(
            "ncm_ace18",
            ValidationStatus.MANUAL_VERIFICATION,
            "\"Código NCM\" is missing from the DJO; cannot query the trade agreement.",
            mercosul_rule=None,
        )

    service = AGREEMENT_SERVICES[agreement]()

    try:
        result = service.query_ncm(ncm)
    except ValueError as exc:
        # e.g. malformed NCM (not exactly 8 digits)
        return make_result(
            "ncm_ace18",
            ValidationStatus.MANUAL_VERIFICATION,
            f"Could not query {agreement} for NCM '{ncm}': {exc}",
            mercosul_rule=None,
        )

    if not result["found"]:
        return make_result(
            "ncm_ace18",
            ValidationStatus.MANUAL_VERIFICATION,
            f"NCM '{ncm}' was not found in {agreement}. Manual verification required — "
            f"the NCM's validity is not assumed either way.",
            mercosul_rule=None,
        )

    return make_result(
        "ncm_ace18",
        ValidationStatus.PASS,
        f"NCM '{ncm}' matched {agreement} rule: {result['mercosul_rule']['raw_rule']}",
        mercosul_rule={
            "matched_ncm_expression": result["mercosul_rule"]["matched_ncm_expression"],
            "raw_rule": result["mercosul_rule"]["raw_rule"],
        },
    )


## Rule 4 — Mandatory fields

`"Valor FOB (USD)"`, `"Unidade de medida"` and `"Descrição do processo
produtivo"` must be present. These are top-level scalar fields, where the
DJO extraction pipeline already collapses a literal `"[Não Informado]"`
into `""` — `is_missing_or_not_informado` still checks for the literal
placeholder too, in case a differently-structured `json_djo` (a future
extractor, or manually-edited data) preserves it instead.


In [7]:
MANDATORY_TOP_LEVEL_FIELDS = ["Valor FOB (USD)", "Unidade de medida", "Descrição do processo produtivo"]


def validate_mandatory_fields(json_djo: Dict[str, Any]) -> Dict[str, Any]:
    missing = [f for f in MANDATORY_TOP_LEVEL_FIELDS if is_missing_or_not_informado(json_djo.get(f))]

    if missing:
        return make_result(
            "mandatory_fields",
            ValidationStatus.FAIL,
            "Missing required fields: " + ", ".join(missing) + ".",
            details={"missing_fields": missing},
        )

    return make_result(
        "mandatory_fields",
        ValidationStatus.PASS,
        "All mandatory fields are present.",
    )


## Rule 5 — Process materials (Gemini semantic matching)

The only rule that genuinely needs language understanding: does every
input named in the free-text `"Descrição do processo produtivo"` show up
(exactly or with reasonable semantic equivalence) in one of the material
tables' `Descrição` cells? The prompt (see `gemini_service.py`) explicitly
instructs the model to extract only *inputs*, not products synthesized
from them, and to never invent a material the document doesn't mention.

If the process description is empty, Rule 4 already reports that as a
missing mandatory field — Rule 5 reports `NOT_APPLICABLE` instead of a
second, redundant failure. If the Gemini call itself fails (no API key,
network error, unparsable response), the result is `MANUAL_VERIFICATION`,
never an assumed PASS or FAIL.


In [8]:
def collect_material_descriptions(materiales: Dict[str, Any]) -> List[Dict[str, str]]:
    """Flattens every material table's Items into a
    [{"table": <table_name>, "descricao": <Descrição value>}, ...] list,
    the input Rule 5's Gemini prompt is built from."""
    collected = []
    for table_name in MATERIAL_TABLE_NAMES:
        table = materiales.get(table_name) or {}
        for item in table.get("Items", []):
            descricao = item.get("Descrição", "")
            if not is_blank(descricao):
                collected.append({"table": table_name, "descricao": descricao})
    return collected


def validate_process_materials(
    json_djo: Dict[str, Any],
    gemini_config: Optional["gemini_service.GeminiConfig"] = None,
) -> Dict[str, Any]:
    descricao_processo = json_djo.get("Descrição do processo produtivo")
    if is_blank(descricao_processo):
        return make_result(
            "process_materials",
            ValidationStatus.NOT_APPLICABLE,
            "\"Descrição do processo produtivo\" is empty; nothing to analyze.",
        )

    materiales = json_djo.get("Materiales") or {}
    material_items = collect_material_descriptions(materiales)

    try:
        llm_result = gemini_service.match_process_materials(
            descricao_processo, material_items, config=gemini_config
        )
    except gemini_service.GeminiServiceError as exc:
        return make_result(
            "process_materials",
            ValidationStatus.MANUAL_VERIFICATION,
            f"Could not run automated material matching: {exc}",
        )

    all_found = bool(llm_result.get("all_inputs_found"))
    not_found = [i["input"] for i in llm_result.get("process_inputs", []) if not i.get("found")]

    status = ValidationStatus.PASS if all_found else ValidationStatus.FAIL
    message = (
        "All process inputs are represented in the material tables."
        if all_found
        else "Some process inputs were not found in the material tables: " + ", ".join(not_found) + "."
    )

    return make_result("process_materials", status, message, details=llm_result)


## Rule 6 — Material table completeness

`originarios_estado_parte_produtor` is the **only** table where `NCM/SH`
and `Valor (US$)` may be `"[Não Informado]"`/empty (Rule 6.1); every other
column, and every column in the other 3 tables, must be present and not
`"[Não Informado]"`. A table with zero `Items` is treated as "does not
exist in this DJO" — `NOT_APPLICABLE`, not a failure (per Rule 6's own
wording): the absence of, say, `terceiros_paises_ptc` is entirely normal
and is not itself a defect.


In [9]:
ITEM_COLUMNS = ["NCM/SH", "Descrição", "País origem", "Valor (US$)", "% s/Valor FOB", "Fornecedor/Fabricante"]

# Rule 6.1: only these two columns, and only in
# "originarios_estado_parte_produtor", may be "[Não Informado]"/empty.
COLUMNS_ALLOWED_NOT_INFORMADO = {"NCM/SH", "Valor (US$)"}


def _validate_table_item(table_name: str, item: Dict[str, str], item_index: int) -> List[str]:
    """Returns human-readable issues for one item; an empty list means the
    item is fully valid for `table_name`'s rules."""
    issues = []
    for column in ITEM_COLUMNS:
        value = item.get(column)
        if is_missing_or_not_informado(value):
            if table_name == "originarios_estado_parte_produtor" and column in COLUMNS_ALLOWED_NOT_INFORMADO:
                continue  # explicitly allowed by Rule 6.1
            issues.append(f"item {item_index}: '{column}' is missing or '{NAO_INFORMADO}'")
    return issues


def validate_table_completeness(materiales: Dict[str, Any]) -> Dict[str, Any]:
    details: Dict[str, Any] = {}
    any_fail = False
    any_checked = False

    for table_name in MATERIAL_TABLE_NAMES:
        table = materiales.get(table_name) or {}
        items = table.get("Items", [])

        if not items:
            details[table_name] = {"status": ValidationStatus.NOT_APPLICABLE.value, "issues": []}
            continue

        any_checked = True
        issues: List[str] = []
        for idx, item in enumerate(items):
            issues.extend(_validate_table_item(table_name, item, idx))

        if issues:
            any_fail = True
            details[table_name] = {"status": ValidationStatus.FAIL.value, "issues": issues}
        else:
            details[table_name] = {"status": ValidationStatus.PASS.value, "issues": []}

    if not any_checked:
        overall_status, message = ValidationStatus.NOT_APPLICABLE, "No material tables with data were present in this DJO."
    elif any_fail:
        overall_status, message = ValidationStatus.FAIL, "One or more material tables have missing/invalid required columns."
    else:
        overall_status, message = ValidationStatus.PASS, "All existing material tables have complete required columns."

    return make_result("table_completeness", overall_status, message, details=details)


## Rule 7 — Material table percentage sums

For every table that actually has data, the sum of its items' `"%
s/Valor FOB"` must numerically match the table's own declared
`"Somatório"`, within `PERCENTAGE_TOLERANCE` percentage points (absorbs
rounding across several line items without masking a real mismatch). Per
the rule's own wording, a table with `Somatório == ""` (i.e. it doesn't
exist) is skipped entirely — not a failure.


In [10]:
def validate_table_percentages(materiales: Dict[str, Any]) -> Dict[str, Any]:
    details: Dict[str, Any] = {}
    any_fail = False
    any_manual = False
    any_checked = False

    for table_name in MATERIAL_TABLE_NAMES:
        table = materiales.get(table_name) or {}
        items = table.get("Items", [])
        somatorio_raw = table.get("Somatório", "")

        if not items or is_blank(somatorio_raw):
            details[table_name] = {"status": ValidationStatus.NOT_APPLICABLE.value}
            continue

        somatorio = parse_percentage(somatorio_raw)
        item_percentages = [parse_percentage(item.get("% s/Valor FOB")) for item in items]

        if somatorio is None or any(p is None for p in item_percentages):
            # A value couldn't be parsed at all — a data problem, not a
            # numeric mismatch. Flag for manual review instead of treating
            # the unparsable value as 0 (which would silently mask it).
            any_checked = True
            any_manual = True
            details[table_name] = {
                "status": ValidationStatus.MANUAL_VERIFICATION.value,
                "declared_somatorio": somatorio_raw,
            }
            continue

        any_checked = True
        calculated_sum = round(sum(item_percentages), 2)
        difference = round(abs(calculated_sum - somatorio), 2)
        passed = difference <= PERCENTAGE_TOLERANCE

        details[table_name] = {
            "status": ValidationStatus.PASS.value if passed else ValidationStatus.FAIL.value,
            "calculated_sum": calculated_sum,
            "declared_somatorio": somatorio,
            "difference": difference,
        }
        if not passed:
            any_fail = True

    if not any_checked:
        overall_status = ValidationStatus.NOT_APPLICABLE
        message = "No material tables with a declared Somatório were present."
    elif any_fail:
        overall_status = ValidationStatus.FAIL
        message = "The sum of '% s/Valor FOB' does not match the declared Somatório for one or more tables."
    elif any_manual:
        overall_status = ValidationStatus.MANUAL_VERIFICATION
        message = "One or more percentage values could not be parsed; manual verification required."
    else:
        overall_status = ValidationStatus.PASS
        message = "The sum of '% s/Valor FOB' matches the declared Somatório for all applicable tables."

    return make_result("table_percentage", overall_status, message, details=details)


## Rule 8 — Preço FOB ceiling

`json_djo["Materiales"]["Preço FOB"]` must not exceed 100%. Parsed with
the same Brazilian-decimal-format helper used for Rule 7. A missing value
is reported as `MANUAL_VERIFICATION` — never silently treated as 0, which
would incorrectly read as an automatic PASS.


In [11]:
def validate_preco_fob(materiales: Dict[str, Any]) -> Dict[str, Any]:
    raw_value = materiales.get("Preço FOB", "")

    if is_missing_or_not_informado(raw_value):
        return make_result(
            "preco_fob",
            ValidationStatus.MANUAL_VERIFICATION,
            "\"Preço FOB\" is missing; cannot validate against the 100% ceiling.",
        )

    value = parse_percentage(raw_value)
    if value is None:
        return make_result(
            "preco_fob",
            ValidationStatus.MANUAL_VERIFICATION,
            f"\"Preço FOB\" value '{raw_value}' could not be parsed as a percentage.",
        )

    if value > 100:
        return make_result(
            "preco_fob",
            ValidationStatus.FAIL,
            f"\"Preço FOB\" is {value}%, which exceeds the 100% ceiling.",
            details={"value": value},
        )

    return make_result(
        "preco_fob",
        ValidationStatus.PASS,
        f"\"Preço FOB\" is {value}%, within the allowed range.",
        details={"value": value},
    )


## Orchestration — run every rule

`validate_djo` is the single entry point the future API layer would call.
It does not know or care how `json_djo` was produced (DJO extraction is a
separate concern — see the top of this notebook) — it just consumes the
structure and the requested `agreement`.

`overall_status` is computed as: `FAIL` if any rule failed, else
`MANUAL_VERIFICATION` if any rule needs manual review, else `PASS`.
`NOT_APPLICABLE` results never affect `overall_status`.


In [12]:
def _overall_status(validations: List[Dict[str, Any]]) -> ValidationStatus:
    statuses = {v["status"] for v in validations}
    if ValidationStatus.FAIL.value in statuses:
        return ValidationStatus.FAIL
    if ValidationStatus.MANUAL_VERIFICATION.value in statuses:
        return ValidationStatus.MANUAL_VERIFICATION
    return ValidationStatus.PASS


def validate_djo(
    json_djo: Dict[str, Any],
    agreement: str = "ACE_18",
    gemini_config: Optional["gemini_service.GeminiConfig"] = None,
) -> Dict[str, Any]:
    """Runs all 8 business rules against `json_djo` and returns the
    structured validation_result described at the top of this notebook.

    `agreement` selects which trade-agreement service backs Rule 3 (see
    AGREEMENT_SERVICES / normalize_agreement) — this is the extension
    point for supporting agreements beyond ACE 18 later, without changing
    any other rule.
    """
    agreement = normalize_agreement(agreement)
    materiales = json_djo.get("Materiales") or {}

    validations = [
        validate_djo_approval(json_djo),
        validate_producer_information(json_djo),
        validate_ncm_ace18(json_djo, agreement),
        validate_mandatory_fields(json_djo),
        validate_process_materials(json_djo, gemini_config),
        validate_table_completeness(materiales),
        validate_table_percentages(materiales),
        validate_preco_fob(materiales),
    ]

    return {
        "overall_status": _overall_status(validations).value,
        "agreement": agreement,
        "validations": validations,
    }


## Presentation layer — Gemini summary (optional)

Turns the already-decided `validation_result` into a short, human-readable
(Portuguese) summary for the future frontend. This step is purely
presentational — see the prompt in `gemini_service.py`, which explicitly
instructs the model not to reinterpret or change any status. If the call
fails for any reason, callers should fall back to displaying the
structured `validation_result` itself, which is always available
regardless of Gemini's availability.


In [13]:
def generate_summary(
    validation_result: Dict[str, Any],
    gemini_config: Optional["gemini_service.GeminiConfig"] = None,
) -> str:
    try:
        return gemini_service.summarize_validation_result(validation_result, config=gemini_config)
    except gemini_service.GeminiServiceError as exc:
        return f"(Summary unavailable: {exc})"


## Example execution

`EXAMPLE_JSON_DJO` below is a previously-extracted, already-validated
`json_djo` sample (produced by `extraction_djo.ipynb` from one of the
sample PDFs in `dados/`) — used here purely to demonstrate the pipeline
end-to-end. No rule above depends on any value from this specific
document; any `json_djo` with the same structure works the same way. In
the real pipeline, `json_djo` would come directly from
`extraction_djo.ipynb`'s `parsear_djo(...)` output instead of a literal
constant.


In [14]:
EXAMPLE_JSON_DJO = {
    "Código da aprovação DJO": "94049000-265640327",
    "Número da DJO": "5640327",
    "Data de apresentação": "23 de Julho de 2026",
    "Razão social do produtor": {
        "Razão social": "ALTENBURG TÊXTIL S.A. 75.293.662/0001-04",
        "CNPJ/CPF": "75.293.662/0001-04",
        "Inscrição Estadual": "251141020",
    },
    "Domicílio legal e parque industrial do produtor": {
        "Endereço": "BR-470, KM 61 - NR.7235 - Bairro BADENFURT - CEP 89070-205 - BLUMENAU - SC - BRASIL",
        "Tel": "4733311516",
        "E-mail": "fabiano@altenburg.com.br",
    },
    "Razão social do exportador": {"Razão social": "", "CNPJ/CPF": "", "Inscrição Estadual": ""},
    "Domicílio legal e parque industrial do exportador": {"Endereço": "", "Tel": "", "E-mail": ""},
    "Código NCM": "9404.90.00",
    "NALADI": "9404.90.00",
    "Denominação comercial do produto a exportar": (
        "Suportes elásticos para camas (somiês); colchões, edredões, almofadas, pufes, "
        "travesseiros e artigos semelhantes. ALMOHADA LEVITARE HOME COLLECTION 50CM X 70CM - "
        "TELA: 100% ALGODÓN, RELLENO: 100% POLIÉSTER"
    ),
    "Valor FOB (USD)": "7,39",
    "Unidade de medida": "PE",
    "Descrição do processo produtivo": (
        "CORTAR TECIDO PARA FORMAR CAPA DO TRAVESSEIRO, COSTURAR CAPA, VIRAR E REVISAR, "
        "APLICAR SOUTACHE E COSTURAR, EMBUTIR E ENCHER COM FIBRA DE POLIÉSTER, FECHAR "
        "TRAVESSEIRO, APLICAR ETIQUETAS, BATER TRAVESSEIRO, EMBALAR, SOLDAR E ENSACAR."
    ),
    "Materiales": {
        "originarios_estado_parte_produtor": {
            "Items": [
                {"NCM/SH": "[Não Informado]", "Descrição": "LINHA T.140", "País origem": "BRASIL",
                 "Valor (US$)": "[Não Informado]", "% s/Valor FOB": "0%",
                 "Fornecedor/Fabricante": "DISPONIBLE À SOLITUD DE LAS AUTORIDADES COMPETENTES"},
                {"NCM/SH": "[Não Informado]", "Descrição": "SOUTACHE ACETINADO", "País origem": "BRASIL",
                 "Valor (US$)": "[Não Informado]", "% s/Valor FOB": "0%",
                 "Fornecedor/Fabricante": "DISPONIBLE À SOLITUD DE LAS AUTORIDADES COMPETENTES"},
                {"NCM/SH": "[Não Informado]", "Descrição": "BOLSA DE PVC", "País origem": "BRASIL",
                 "Valor (US$)": "[Não Informado]", "% s/Valor FOB": "0%",
                 "Fornecedor/Fabricante": "DISPONIBLE À SOLITUD DE LAS AUTORIDADES COMPETENTES"},
                {"NCM/SH": "[Não Informado]", "Descrição": "ENCARTE", "País origem": "BRASIL",
                 "Valor (US$)": "[Não Informado]", "% s/Valor FOB": "0%",
                 "Fornecedor/Fabricante": "DISPONIBLE À SOLITUD DE LAS AUTORIDADES COMPETENTES"},
                {"NCM/SH": "[Não Informado]", "Descrição": "ETIQUETA DE COMPOSIÇÃO", "País origem": "BRASIL",
                 "Valor (US$)": "[Não Informado]", "% s/Valor FOB": "0%",
                 "Fornecedor/Fabricante": "DISPONIBLE À SOLITUD DE LAS AUTORIDADES COMPETENTES"},
                {"NCM/SH": "[Não Informado]", "Descrição": "ETIQUETA ADESIVA", "País origem": "BRASIL",
                 "Valor (US$)": "[Não Informado]", "% s/Valor FOB": "0%",
                 "Fornecedor/Fabricante": "DISPONIBLE À SOLITUD DE LAS AUTORIDADES COMPETENTES"},
            ],
            "Somatório": "0%",
        },
        "originarios_outros_estados_partes": {"Items": [], "Somatório": ""},
        "nao_originarios": {
            "Items": [
                {"NCM/SH": "5208.22.00", "Descrição": "TECIDO PERCAL 233 FIOS 134 G/M2 100% ALGODÃO",
                 "País origem": "CHINA", "Valor (US$)": "0,97", "% s/Valor FOB": "13,13%",
                 "Fornecedor/Fabricante": "DISPONIBLE À SOLITUD DE LAS AUTORIDADES COMPETENTES"},
                {"NCM/SH": "5503.20.90", "Descrição": "FIBRA DE POLIÉSTER", "País origem": "CHINA",
                 "Valor (US$)": "1,45", "% s/Valor FOB": "19,62%",
                 "Fornecedor/Fabricante": "DISPONIBLE À SOLITUD DE LAS AUTORIDADES COMPETENTES"},
            ],
            "Somatório": "32,75%",
        },
        "terceiros_paises_ptc": {"Items": [], "Somatório": ""},
        "Porcentagem Total de Matérias Primas, Componentes ou Partes": "32,75%",
        "Valor agregado no processo Industrial (Deduzidos os tributos restituídos ou a restituir em caso de exportação)": "67,25%",
        "Preço FOB": "100%",
    },
    "Observações": "",
}

validation_result = validate_djo(EXAMPLE_JSON_DJO, agreement="ACE_18")
print(json.dumps(validation_result, indent=2, ensure_ascii=False))


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{
  "overall_status": "MANUAL_VERIFICATION",
  "agreement": "ACE_18",
  "validations": [
    {
      "rule": "djo_approval",
      "status": "PASS",
      "message": "The DJO has already been approved.",
      "details": {
        "codigo_aprovacao": "94049000-265640327",
        "data_apresentacao": "23 de Julho de 2026"
      }
    },
    {
      "rule": "producer_information",
      "status": "PASS",
      "message": "Producer identity and domicile are present."
    },
    {
      "rule": "ncm_ace18",
      "status": "PASS",
      "message": "NCM '9404.90.00' matched ACE_18 rule: MP ou MaxMNO 45%",
      "mercosul_rule": {
        "matched_ncm_expression": "ex Capítulo 94(*)",
        "raw_rule": "MP ou MaxMNO 45%"
      }
    },
    {
      "rule": "mandatory_fields",
      "status": "PASS",
      "message": "All mandatory fields are present."
    },
    {
      "rule": "process_materials",
      "status": "MANUAL_VERIFICATION",
      "message": "Could not run automated material ma

## ACE 18 Origin-Rule Decision Engine (flowchart)

This section implements the **last stage** of the pipeline: the official
ACE 18 (MERCOSUL) rules-of-origin flowchart
(`RedeCIN_COD_Anexo1_Fluxograma_ACE18_NovoRegimeOrigemMercosul.pdf`), on top
of — not instead of — everything above. It **reuses** `validate_djo` (does
not re-validate the DJO) and `AGREEMENT_SERVICES`/`ACE18Service` (does not
re-query or re-cache ACE 18 data).

```
json_djo + product_category (+ article14_compliant)
        ↓
validate_djo(...)                      <- reused as-is, not duplicated
        ↓ (only if overall_status == PASS)
evaluate_origin_rule(...)              <- the flowchart itself
        ↓
generate_full_report(...)              <- + Gemini validation summary,
                                           with the flowchart's plain-language
                                           explanation appended after it
```

**Three places where the flowchart's own wording had to be mapped onto the
structured `json_djo` fields — documented here explicitly rather than left
implicit, per the instruction to flag rather than silently guess:**

1. *"O produto é totalmente elaborado... (Art. 5°)?"* and *"...elaborado
   exclusivamente a partir de materiais originários?"* — the DJO JSON has
   no field describing production process against Art. 5 wording. Per this
   component's own spec (section 3), this pair of questions is answered
   from which originating-material tables are non-empty:
   `originarios_estado_parte_produtor` only → **Rule A**;
   both `originarios_estado_parte_produtor` and
   `originarios_outros_estados_partes` non-empty → **Rule B**; any other
   combination → `MANUAL_VERIFICATION` (never guessed).
2. *"O produto possui apenas as operações... insuficientes (Art. 8°)?"* —
   likewise has no direct field. Per spec (section 4), an empty
   `"Descrição do processo produtivo"` is treated as equivalent to failing
   this check → `DOES_NOT_CONFER_ORIGIN`.
3. *"O produto cumpre com a regra de 'De minimis' (Art. 6°)?"* — this
   question requires a numeric tolerance threshold that appears **nowhere**
   in the flowchart PDF text or in the ACE 18 rule cache; only a boolean
   (`de_minimis_applies`, already extracted by `ace18_service.parse_mercosul_rule`
   from phrases like "Não se aplica de minimis") is available. So: if the
   rule text explicitly excludes de minimis (`de_minimis_applies is False`)
   the answer is a firm "não" → `DOES_NOT_CONFER_ORIGIN`; otherwise the
   question genuinely cannot be answered deterministically →
   `MANUAL_VERIFICATION` (never assumed to pass). **This is a real gap in
   the available data, not a design shortcut — flagging it explicitly
   rather than inventing a percentage.**

The deterministic rule-condition parser (`parse_rule_conditions`) was
validated against all 584 rows / 25 distinct `raw_rule` texts in the actual
`reos_ace18_cache.json`, not invented patterns — see the parsing/evaluator
cells below for what each pattern looks like in practice.


### Deterministic condition evaluators (MP / MSP / MaxMNO / excluded positions)

Each evaluator takes the product's `Código NCM` and the pooled
`nao_originarios` + `terceiros_paises_ptc` items, and returns
`(result, reason)` where `result` is `True`/`False`/`None` — `None` means
"could not be validated with certainty" (e.g. a missing/malformed NCM in a
material row), which must propagate as `MANUAL_VERIFICATION`, never as an
assumed pass.


In [15]:
# =============================================================================
# Rule-text parsing: split mercosul_rule.raw_rule into atomic conditions +
# the logical operator joining them. Purely syntactic — no evaluation here.
# =============================================================================
import re

# Products this flowchart applies to (spec section 2): only industrial and
# game/assortment products; automotive is explicitly out of scope.
PRODUCT_CATEGORIES = {"industrial", "automotive", "game"}


def parse_rule_conditions(raw_rule: str) -> Dict[str, Any]:
    """Splits a mercosul_rule raw_rule string into atomic conditions plus
    the operator joining them, e.g.:

        "MP ou MaxMNO 45%"
            -> operator "OR", conditions [MP, MAXMNO(45)]
        "MP, exceto das posições 7206 a 7217"
            -> operator "AND", conditions [MP, EXCLUDED_POSITIONS(7206-7217)]
        "MP mais MaxMNO 45%. Não se aplica de minimis..."
            -> operator "AND", conditions [MP, MAXMNO(45)], de_minimis_applies=False

    Validated against all 25 distinct raw_rule texts found in the real
    reos_ace18_cache.json (584 rows). Known limitation: only ONE operator
    level is detected per rule (first "ou", else "mais", else a comma for
    the "MP, exceto..." exception pattern) — no rule in that corpus mixes
    operators, so this single-level split is sufficient for the observed
    data; a rule mixing "ou" and "mais" together would not be split
    correctly.

    Returns {"operator": "SINGLE"|"OR"|"AND", "conditions": [...],
    "de_minimis_applies": bool}.
    """
    texto = raw_rule.strip()

    # "Não se aplica de minimis..." is metadata about Art. 6 applicability,
    # not a pass/fail condition of the origin requirement itself (reuses
    # the same phrase ace18_service.parse_mercosul_rule already looks for)
    # — strip it before splitting into conditions.
    de_minimis_applies = "não se aplica de minimis" not in texto.lower()
    texto_sem_minimis = re.split(r"\.\s*Não se aplica de minimis", texto, flags=re.IGNORECASE)[0].strip()

    if re.search(r"\bou\b", texto_sem_minimis, re.IGNORECASE):
        operator = "OR"
        segmentos = re.split(r"\bou\b", texto_sem_minimis, flags=re.IGNORECASE)
    elif re.search(r"\bmais\b", texto_sem_minimis, re.IGNORECASE):
        operator = "AND"
        segmentos = re.split(r"\bmais\b", texto_sem_minimis, flags=re.IGNORECASE)
    elif "," in texto_sem_minimis:
        operator = "AND"
        segmentos = texto_sem_minimis.split(",", 1)
    else:
        operator = "SINGLE"
        segmentos = [texto_sem_minimis]

    condicoes = [_classify_condition(seg.strip(" .")) for seg in segmentos if seg.strip(" .")]
    return {"operator": operator, "conditions": condicoes, "de_minimis_applies": de_minimis_applies}


def _classify_condition(segment: str) -> Dict[str, Any]:
    """Classifies one already-split segment into a known deterministic
    condition type, or UNKNOWN (free text — handled later via Gemini)."""
    seg = segment.strip()
    seg_upper = seg.upper()

    if seg_upper == "MSP":
        return {"type": "MSP", "text": seg}
    if seg_upper == "MP":
        return {"type": "MP", "text": seg}

    m = re.match(r"^MaxMNO\s*(\d+(?:[.,]\d+)?)\s*%$", seg, re.IGNORECASE)
    if m:
        return {"type": "MAXMNO", "text": seg, "max_percentage": float(m.group(1).replace(",", "."))}

    m = re.match(r"^exceto\s+das\s+posições\s+(\d{4})\s+a\s+(\d{4})$", seg, re.IGNORECASE)
    if m:
        return {"type": "EXCLUDED_POSITIONS", "text": seg,
                "range_start": int(m.group(1)), "range_end": int(m.group(2))}

    return {"type": "UNKNOWN", "text": seg}


def _first_n_digits(ncm: Any, n: int) -> Optional[str]:
    digits = re.sub(r"\D", "", str(ncm or ""))
    return digits[:n] if len(digits) >= n else None


def evaluate_mp_condition(product_ncm: str, tables_items: List[Dict[str, Any]]) -> tuple:
    """MP (Mudança de Partida / change of tariff heading): the product's
    4-digit heading must differ from the 4-digit heading of every
    non-originating/third-country material."""
    produto_4 = _first_n_digits(product_ncm, 4)
    if produto_4 is None:
        return None, "Código NCM do produto inválido/ausente; não é possível validar MP."

    conflitos = [item.get("NCM/SH") for item in tables_items if _first_n_digits(item.get("NCM/SH"), 4) == produto_4]
    inconclusivos = [item.get("NCM/SH") for item in tables_items if _first_n_digits(item.get("NCM/SH"), 4) is None]

    if inconclusivos:
        return None, f"NCM(s) inválido(s)/ausente(s) nos materiais não originários: {inconclusivos}; MP não pôde ser validada com certeza."
    if conflitos:
        return False, f"Material(is) não originário(s) compartilham a partida (4 dígitos) do produto ({produto_4}): {conflitos}."
    return True, f"Nenhum material não originário compartilha a partida (4 dígitos) do produto ({produto_4})."


def evaluate_msp_condition(product_ncm: str, tables_items: List[Dict[str, Any]]) -> tuple:
    """MSP (Mudança de Subpartida / change of tariff subheading): same as
    MP but comparing 6-digit subheadings."""
    produto_6 = _first_n_digits(product_ncm, 6)
    if produto_6 is None:
        return None, "Código NCM do produto inválido/ausente; não é possível validar MSP."

    conflitos = [item.get("NCM/SH") for item in tables_items if _first_n_digits(item.get("NCM/SH"), 6) == produto_6]
    inconclusivos = [item.get("NCM/SH") for item in tables_items if _first_n_digits(item.get("NCM/SH"), 6) is None]

    if inconclusivos:
        return None, f"NCM(s) inválido(s)/ausente(s) nos materiais não originários: {inconclusivos}; MSP não pôde ser validada com certeza."
    if conflitos:
        return False, f"Material(is) não originário(s) compartilham a subpartida (6 dígitos) do produto ({produto_6}): {conflitos}."
    return True, f"Nenhum material não originário compartilha a subpartida (6 dígitos) do produto ({produto_6})."


def evaluate_maxmno_condition(max_percentage: float, tables_items: List[Dict[str, Any]]) -> tuple:
    """MaxMNO x%: the sum of '% s/Valor FOB' across the non-originating /
    third-country tables must not exceed `max_percentage`. Uses the same
    `parse_percentage` helper defined earlier for Rules 7/8, so the same
    Brazilian-decimal-format handling and [Não Informado]-awareness apply."""
    valores = [parse_percentage(item.get("% s/Valor FOB")) for item in tables_items]
    if any(v is None for v in valores):
        return None, "Um ou mais valores de '% s/Valor FOB' não puderam ser interpretados; não foi possível calcular o total para MaxMNO."
    total = round(sum(valores), 2)
    if total <= max_percentage:
        return True, f"Soma de '% s/Valor FOB' dos materiais não originários ({total}%) está dentro do limite MaxMNO ({max_percentage}%)."
    return False, f"Soma de '% s/Valor FOB' dos materiais não originários ({total}%) excede o limite MaxMNO ({max_percentage}%)."


def evaluate_excluded_positions_condition(range_start: int, range_end: int, tables_items: List[Dict[str, Any]]) -> tuple:
    """'exceto das posições X a Y': none of the non-originating/third-country
    materials' 4-digit headings may fall inside [range_start, range_end]."""
    inconclusivos = [item.get("NCM/SH") for item in tables_items if _first_n_digits(item.get("NCM/SH"), 4) is None]
    if inconclusivos:
        return None, f"NCM(s) inválido(s)/ausente(s): {inconclusivos}; a exclusão de posições não pôde ser validada com certeza."

    violacoes = [
        item.get("NCM/SH") for item in tables_items
        if range_start <= int(_first_n_digits(item.get("NCM/SH"), 4)) <= range_end
    ]
    if violacoes:
        return False, f"Material(is) não originário(s) pertencem à faixa excluída {range_start}-{range_end}: {violacoes}."
    return True, f"Nenhum material não originário pertence à faixa excluída de posições {range_start}-{range_end}."


### Gemini fallback + condition-set evaluation

`evaluate_rule_conditions` combines every condition in a parsed rule
according to its operator (`OR`/`AND`/`SINGLE`). Deterministic conditions
(MP/MSP/MAXMNO/EXCLUDED_POSITIONS) are evaluated first; Gemini
(`evaluate_unknown_condition_with_gemini`) is invoked **only** for the
remaining free-text (`UNKNOWN`) conditions — e.g. "Reação química",
"Processo biotecnológico", or a fully free-text rule like "Devem ser
elaborados a partir de leite produzido nos Estados Partes" — and **only**
when the deterministic ones could not already decide the outcome alone
(short-circuit: an `OR` with a deterministic `True`, or an `AND` with a
deterministic `False`, never touches Gemini). All remaining `UNKNOWN`
segments are bundled into a **single** Gemini call (asking "does the
product satisfy *any* of these" for `OR`, or evaluated as one combined
requirement otherwise) rather than one call per segment.


In [16]:
def evaluate_unknown_condition_with_gemini(
    condition_texts: List[str],
    descricao_processo: str,
    tables_items: List[Dict[str, Any]],
    gemini_config: Optional["gemini_service.GeminiConfig"] = None,
) -> tuple:
    """Delegates one or more UNKNOWN (free-text) rule conditions to Gemini.
    Returns (result, reason); result is True/False/None (None = Gemini
    failed or itself returned an inconclusive answer)."""
    material_items_for_prompt = [
        {
            "table": item.get("_table", "?"),
            "ncm": item.get("NCM/SH", ""),
            "pais_origem": item.get("País origem", ""),
            "descricao": item.get("Descrição", ""),
        }
        for item in tables_items
    ]
    try:
        resultado = gemini_service.evaluate_rule_condition(
            condition_texts, descricao_processo, material_items_for_prompt, config=gemini_config,
        )
        return resultado.get("result"), resultado.get("reasoning", "")
    except gemini_service.GeminiServiceError as exc:
        return None, f"Falha ao consultar o Gemini para avaliação semântica: {exc}"


def evaluate_rule_conditions(
    parsed_rule: Dict[str, Any],
    product_ncm: str,
    tables_items: List[Dict[str, Any]],
    descricao_processo: str,
    gemini_config: Optional["gemini_service.GeminiConfig"] = None,
) -> tuple:
    """Evaluates every condition in `parsed_rule`, combined via its
    operator. Returns (result, condition_traces): result is True/False/None
    (None = inconclusive -> MANUAL_VERIFICATION upstream). Each entry in
    condition_traces is {"condition", "type", "result", "reason", "used_gemini"}."""
    operator = parsed_rule["operator"]
    conditions = parsed_rule["conditions"]

    traces: List[Dict[str, Any]] = []
    deterministic_results: List[Optional[bool]] = []

    for cond in conditions:
        if cond["type"] == "MP":
            result, reason = evaluate_mp_condition(product_ncm, tables_items)
        elif cond["type"] == "MSP":
            result, reason = evaluate_msp_condition(product_ncm, tables_items)
        elif cond["type"] == "MAXMNO":
            result, reason = evaluate_maxmno_condition(cond["max_percentage"], tables_items)
        elif cond["type"] == "EXCLUDED_POSITIONS":
            result, reason = evaluate_excluded_positions_condition(cond["range_start"], cond["range_end"], tables_items)
        else:
            continue  # UNKNOWN: handled below, only if the operator still needs it
        traces.append({"condition": cond["text"], "type": cond["type"], "result": result, "reason": reason, "used_gemini": False})
        deterministic_results.append(result)

    # Short-circuit: can the operator already be decided from the
    # deterministic conditions alone, without touching UNKNOWN/Gemini ones?
    if operator == "OR" and any(r is True for r in deterministic_results):
        return True, traces
    if operator in ("AND", "SINGLE") and any(r is False for r in deterministic_results):
        return False, traces

    unknown_conditions = [c for c in conditions if c["type"] == "UNKNOWN"]

    if not unknown_conditions:
        if any(r is None for r in deterministic_results):
            return None, traces
        return (any(deterministic_results) if operator == "OR" else all(deterministic_results)), traces

    gemini_result, gemini_reason = evaluate_unknown_condition_with_gemini(
        [c["text"] for c in unknown_conditions], descricao_processo, tables_items, gemini_config,
    )
    traces.append({
        "condition": " | ".join(c["text"] for c in unknown_conditions),
        "type": "UNKNOWN", "result": gemini_result, "reason": gemini_reason, "used_gemini": True,
    })

    if gemini_result is None or any(r is None for r in deterministic_results):
        return None, traces

    if operator == "OR":
        combined = any(deterministic_results) or gemini_result
    else:  # AND / SINGLE
        combined = all(deterministic_results) and gemini_result

    return combined, traces


### Origin-rule decision engine orchestration

`evaluate_origin_rule` walks the flowchart top to bottom, recording one
`decision_trace` entry per question. `product_category` is checked first —
`automotive` short-circuits to `NOT_APPLICABLE` before even running the
(potentially Gemini-backed) base DJO validation, since the flowchart never
applies to it regardless of DJO state. For `industrial`/`game`, the base
validation gate runs next, exactly as spec'd: a `FAIL` maps to
`CORRECT_DJO` ("Corrigir Declaração"), a `MANUAL_VERIFICATION` propagates
as-is, and only `PASS` lets the flowchart continue.


In [17]:
def determine_origin_rule_simple(materiales: Dict[str, Any]) -> tuple:
    """No third-country materials: Rule A/B is decided purely from which
    originating-material tables are non-empty (see this section's opening
    markdown cell for why — the DJO JSON has no direct Art. 5 field)."""
    tem_estado_parte = len((materiales.get("originarios_estado_parte_produtor") or {}).get("Items", [])) > 0
    tem_outros_estados = len((materiales.get("originarios_outros_estados_partes") or {}).get("Items", [])) > 0

    if tem_estado_parte and not tem_outros_estados:
        return "ORIGIN_RULE_A", "A", "Somente 'originarios_estado_parte_produtor' contém materiais -> Norma de Origem A."
    if tem_estado_parte and tem_outros_estados:
        return "ORIGIN_RULE_B", "B", "'originarios_estado_parte_produtor' e 'originarios_outros_estados_partes' contêm materiais -> Norma de Origem B."
    return "MANUAL_VERIFICATION", None, (
        f"Combinação de tabelas originárias não prevista (estado_parte_produtor="
        f"{'presente' if tem_estado_parte else 'vazia'}, outros_estados_partes="
        f"{'presente' if tem_outros_estados else 'vazia'}) — verificação manual necessária."
    )


STATUS_LABELS = {
    "ORIGIN_RULE_A": "Norma de Origem A (produto elaborado exclusivamente a partir de materiais originários / Art. 5°)",
    "ORIGIN_RULE_B": "Norma de Origem B (produto elaborado no território dos Estados Partes a partir de materiais originários)",
    "ORIGIN_RULE_C": "Norma de Origem C (produto cumpre o requisito específico de origem do ACE 18)",
    "DOES_NOT_CONFER_ORIGIN": "o produto NÃO confere origem",
    "CORRECT_DJO": "é necessário corrigir a declaração (DJO) antes de prosseguir",
    "MANUAL_VERIFICATION": "é necessária verificação manual",
    "NOT_APPLICABLE": "este fluxograma não se aplica a este produto",
}


def build_simple_explanation(trace: List[Dict[str, Any]], final_status: str, closing_sentence: str) -> str:
    """Deterministic (non-Gemini) plain-language narration of the decision
    trace. Kept deterministic — not Gemini-generated — specifically so it
    can never contradict the trace it is built from (spec section 13)."""
    linhas = []
    for e in trace:
        r = e["result"]
        r_str = "Sim" if r is True else "Não" if r is False else "Indeterminado" if r is None else str(r)
        linhas.append(f"{e['step']}. {e['question']} -> {r_str}. {e['reason']}")
    corpo = "\n".join(linhas)
    rotulo = STATUS_LABELS.get(final_status, final_status)
    return f"Sequência de decisões do fluxograma ACE 18:\n{corpo}\n\nConclusão: {rotulo}. {closing_sentence}"


def _origin_result(
    validation_result: Optional[Dict[str, Any]],
    final_status: str,
    applicable_rule: Optional[str],
    trace: List[Dict[str, Any]],
    closing_sentence: str,
    ncm_rule_info: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    return {
        "validation_result": validation_result,
        "ncm_rule_info": ncm_rule_info,
        "applicable_origin_rule": applicable_rule,
        "final_status": final_status,
        "decision_trace": trace,
        "simple_explanation": build_simple_explanation(trace, final_status, closing_sentence),
    }


def evaluate_origin_rule(
    json_djo: Dict[str, Any],
    product_category: str,
    article14_compliant: Optional[bool] = None,
    validation_result: Optional[Dict[str, Any]] = None,
    agreement: str = "ACE_18",
    gemini_config: Optional["gemini_service.GeminiConfig"] = None,
) -> Dict[str, Any]:
    """Runs the ACE 18 origin-rule flowchart on top of the existing DJO
    validation. `product_category` (industrial|automotive|game) and, for
    games, `article14_compliant` (True/False/None) are supplied externally
    — never inferred (spec section 2). Pass a pre-computed
    `validation_result` to avoid recomputing it (e.g. from
    `generate_full_report`)."""
    trace: List[Dict[str, Any]] = []
    step_counter = {"n": 0}

    def add_step(question, input_used, result, reason):
        step_counter["n"] += 1
        entry = {"step": step_counter["n"], "question": question, "input_used": input_used, "result": result, "reason": reason}
        trace.append(entry)
        return entry

    # Category is checked FIRST: automotive is entirely out of scope for
    # this flowchart, so it short-circuits before even running the base
    # DJO validation — running (Gemini-backed) validation for a product
    # this flowchart never applies to would be wasted work and would
    # otherwise surface a nonsensical CORRECT_DJO/MANUAL_VERIFICATION for
    # an out-of-scope product.
    if product_category not in PRODUCT_CATEGORIES:
        raise ValueError(f"product_category inválido: {product_category!r}. Esperado um de {PRODUCT_CATEGORIES}.")
    add_step("Qual a categoria do produto?", {"product_category": product_category}, product_category,
              "Categoria informada externamente (não inferida).")
    if product_category == "automotive":
        return _origin_result(None, "NOT_APPLICABLE", None, trace,
                               "Produtos automotivos estão fora do escopo deste fluxograma.")

    if validation_result is None:
        validation_result = validate_djo(json_djo, agreement=agreement, gemini_config=gemini_config)

    add_step(
        "A Declaração Juramentada de Origem contém as informações obrigatórias (validação existente)?",
        {"overall_status": validation_result["overall_status"]},
        validation_result["overall_status"] == "PASS",
        f"Status geral da validação existente: {validation_result['overall_status']}.",
    )
    if validation_result["overall_status"] == "FAIL":
        return _origin_result(validation_result, "CORRECT_DJO", None, trace,
                               "A DJO não passou na validação obrigatória existente; corrija a declaração antes de prosseguir.")
    if validation_result["overall_status"] == "MANUAL_VERIFICATION":
        return _origin_result(validation_result, "MANUAL_VERIFICATION", None, trace,
                               "A validação existente da DJO exige verificação manual antes de prosseguir.")

    materiales = json_djo.get("Materiales") or {}
    nao_originarios_items = (materiales.get("nao_originarios") or {}).get("Items", [])
    terceiros_items = (materiales.get("terceiros_paises_ptc") or {}).get("Items", [])
    tem_terceiros = bool(nao_originarios_items) or bool(terceiros_items)

    add_step(
        "O produto possui materiais originários de terceiros países ('nao_originarios' / 'terceiros_paises_ptc')?",
        {"nao_originarios_items": len(nao_originarios_items), "terceiros_paises_ptc_items": len(terceiros_items)},
        tem_terceiros,
        f"'nao_originarios' com {len(nao_originarios_items)} item(ns), 'terceiros_paises_ptc' com {len(terceiros_items)} item(ns).",
    )

    if not tem_terceiros:
        status, regra, motivo = determine_origin_rule_simple(materiales)
        add_step(
            "Qual tabela de materiais originários está preenchida?",
            {"originarios_estado_parte_produtor_items": len((materiales.get("originarios_estado_parte_produtor") or {}).get("Items", [])),
             "originarios_outros_estados_partes_items": len((materiales.get("originarios_outros_estados_partes") or {}).get("Items", []))},
            regra, motivo,
        )
        return _origin_result(validation_result, status, regra, trace, motivo)

    descricao_processo = json_djo.get("Descrição do processo produtivo")
    tem_descricao = not is_blank(descricao_processo)
    add_step(
        "A 'Descrição do processo produtivo' está preenchida?", {"descricao_presente": tem_descricao}, tem_descricao,
        "Sem a descrição do processo produtivo não é possível avaliar se as operações são suficientes para conferir origem (Art. 8°).",
    )
    if not tem_descricao:
        return _origin_result(validation_result, "DOES_NOT_CONFER_ORIGIN", None, trace,
                               "Produto não confere origem: descrição do processo produtivo ausente.")

    ncm = json_djo.get("Código NCM")
    ncm_query = None
    if not is_blank(ncm):
        service = AGREEMENT_SERVICES[normalize_agreement(agreement)]()
        ncm_query = service.query_ncm(ncm)
    ncm_encontrada = bool(ncm_query and ncm_query.get("found"))

    add_step(
        "A NCM está presente na lista de itens sujeitos a requisitos específicos de origem (base ACE 18)?",
        {"Código NCM": ncm}, ncm_encontrada,
        (f"Regra Mercosul encontrada: {ncm_query['mercosul_rule']['raw_rule']!r}." if ncm_encontrada
         else "NCM ausente na DJO ou não encontrada na base ACE 18 (Apêndice II)."),
    )
    if not ncm_encontrada:
        return _origin_result(validation_result, "DOES_NOT_CONFER_ORIGIN", None, trace,
                               "Produto não confere origem: NCM não localizada na base de regras do ACE 18.", ncm_rule_info=ncm_query)

    mercosul_rule = ncm_query["mercosul_rule"]
    raw_rule = mercosul_rule["raw_rule"]

    if product_category == "game":
        add_step("O produto cumpre com a regra de Jogos e Sortidos, definida no Artigo 14°?",
                  {"article14_compliant": article14_compliant}, article14_compliant,
                  "Resposta fornecida externamente pelo usuário." if article14_compliant is not None else "Resposta ainda não fornecida pelo usuário.")
        if article14_compliant is None:
            return _origin_result(validation_result, "MANUAL_VERIFICATION", None, trace,
                                   "Produto classificado como jogo/sortido: informe se cumpre a regra do Artigo 14° antes de prosseguir.",
                                   ncm_rule_info=ncm_query)
        if article14_compliant is False:
            return _origin_result(validation_result, "DOES_NOT_CONFER_ORIGIN", None, trace,
                                   "Produto não confere origem: não cumpre a regra de Jogos e Sortidos do Artigo 14°.",
                                   ncm_rule_info=ncm_query)

    parsed_rule = parse_rule_conditions(raw_rule)
    tagged_items = (
        [{"_table": "nao_originarios", **item} for item in nao_originarios_items] +
        [{"_table": "terceiros_paises_ptc", **item} for item in terceiros_items]
    )
    complies, condition_traces = evaluate_rule_conditions(parsed_rule, ncm, tagged_items, descricao_processo, gemini_config)

    for ct in condition_traces:
        add_step(
            f"A condição '{ct['condition']}' (operador {parsed_rule['operator']}) é satisfeita?",
            {"raw_rule": raw_rule}, ct["result"],
            ct["reason"] + (" [avaliado via Gemini]" if ct["used_gemini"] else ""),
        )

    add_step(
        "O produto cumpre com o requisito de origem (combinação das condições acima)?",
        {"raw_rule": raw_rule, "operator": parsed_rule["operator"]}, complies,
        "Resultado combinado das condições acima segundo o operador lógico identificado." if complies is not None
        else "Uma ou mais condições não puderam ser avaliadas deterministicamente nem via Gemini.",
    )

    if complies is True:
        return _origin_result(validation_result, "ORIGIN_RULE_C", "C", trace,
                               "Produto confere origem pela Norma C: cumpre o requisito específico de origem do ACE 18 para sua NCM.",
                               ncm_rule_info=ncm_query)
    if complies is None:
        return _origin_result(validation_result, "MANUAL_VERIFICATION", None, trace,
                               "Não foi possível determinar (nem deterministicamente, nem via Gemini) se o produto cumpre o requisito de origem.",
                               ncm_rule_info=ncm_query)

    de_minimis_applies = parsed_rule["de_minimis_applies"]
    add_step(
        "O produto cumpre com a regra de 'De minimis' definida no Artigo 6°?",
        {"de_minimis_applies_per_rule_text": de_minimis_applies}, None if de_minimis_applies else False,
        ("A regra explicitamente declara que 'de minimis' não se aplica." if not de_minimis_applies else
         "A regra não exclui 'de minimis', mas não há um percentual de tolerância disponível nos dados para avaliar esta condição deterministicamente."),
    )
    if not de_minimis_applies:
        return _origin_result(validation_result, "DOES_NOT_CONFER_ORIGIN", None, trace,
                               "Produto não confere origem: não cumpre o requisito específico de origem, e a regra aplicável exclui explicitamente a tolerância de minimis (Art. 6°).",
                               ncm_rule_info=ncm_query)
    return _origin_result(validation_result, "MANUAL_VERIFICATION", None, trace,
                           "Produto não cumpre o requisito específico de origem; a tolerância de minimis (Art. 6°) poderia se aplicar, mas o percentual de tolerância não está disponível nos dados — verificação manual necessária.",
                           ncm_rule_info=ncm_query)


### Final combined report

`generate_full_report` is the single entry point a future API endpoint
would call: it runs the origin-rule engine (which itself runs the base
`validate_djo`), then combines TWO Gemini-authored paragraphs — the
original, unchanged DJO validation summary (`generate_summary`) and a new
paragraph narrating the origin-rule result and the flowchart path it took
(`generate_origin_summary`) — into one consistent-sounding summary. The
deterministic, trace-derived `simple_explanation` is still returned as its
own field for audit/machine use.


In [18]:
def generate_origin_summary(
    origin_result: Dict[str, Any],
    gemini_config: Optional["gemini_service.GeminiConfig"] = None,
) -> str:
    """Same graceful-fallback pattern as generate_summary above, but for
    the origin-rule result: narrates final_status/applicable_origin_rule
    and the flowchart path (decision_trace), in the same presentational
    style as generate_summary, so the two concatenate into one consistent
    summary (see generate_full_report). Falls back to the deterministic
    simple_explanation field if Gemini is unavailable."""
    try:
        return gemini_service.summarize_origin_rule_result(origin_result, config=gemini_config)
    except gemini_service.GeminiServiceError as exc:
        return f"(Resumo da norma de origem indisponível via Gemini: {exc})\n\n{origin_result['simple_explanation']}"


def generate_full_report(
    json_djo: Dict[str, Any],
    product_category: str,
    article14_compliant: Optional[bool] = None,
    agreement: str = "ACE_18",
    gemini_config: Optional["gemini_service.GeminiConfig"] = None,
) -> Dict[str, Any]:
    """Full pipeline entry point: base DJO validation + ACE 18 origin-rule
    flowchart + Gemini summary, combined into one structured report.

    Returns everything `evaluate_origin_rule` returns
    (validation_result, ncm_rule_info, applicable_origin_rule, final_status,
    decision_trace, simple_explanation), plus:
        "gemini_validation_summary": the Gemini-authored summary of
            validation_result (identical to calling generate_summary
            directly),
        "gemini_origin_summary": the Gemini-authored narration of the
            origin-rule result + flowchart path,
        "combined_report_text": the two summaries above, concatenated.
    """
    origin_result = evaluate_origin_rule(
        json_djo, product_category, article14_compliant=article14_compliant,
        agreement=agreement, gemini_config=gemini_config,
    )

    if origin_result["validation_result"] is not None:
        gemini_validation_summary = generate_summary(origin_result["validation_result"], gemini_config)
        gemini_origin_summary = generate_origin_summary(origin_result, gemini_config)
        combined_text = f"{gemini_validation_summary}\n\n{gemini_origin_summary}"
    else:
        # automotive: base validation was never run (out of scope) - nothing to summarize
        gemini_validation_summary = "(Resumo não gerado: fluxograma não aplicável a este produto.)"
        gemini_origin_summary = origin_result["simple_explanation"]
        combined_text = f"{gemini_validation_summary}\n\n{gemini_origin_summary}"

    return {
        **origin_result,
        "gemini_validation_summary": gemini_validation_summary,
        "gemini_origin_summary": gemini_origin_summary,
        "combined_report_text": combined_text,
    }


## Example execution — origin-rule engine

Reuses `EXAMPLE_JSON_DJO` from the earlier example. Its NCM (`9404.90.00`)
maps to ACE 18 rule `"MP ou MaxMNO 45%"`; its non-originating materials
(headings `5208`/`5503`) don't share a heading with the product (`9404`),
so `MP` passes deterministically and the flowchart reaches **Rule C**
without ever needing Gemini for the origin-rule step itself (Rule 5's own
Gemini call, inside the base validation, still applies as before).


In [19]:
full_report = generate_full_report(EXAMPLE_JSON_DJO, product_category="industrial")

print("final_status:", full_report["final_status"])
print("applicable_origin_rule:", full_report["applicable_origin_rule"])
print()
print(json.dumps(full_report["decision_trace"], indent=2, ensure_ascii=False))
print()
print(full_report["combined_report_text"])

final_status: MANUAL_VERIFICATION
applicable_origin_rule: None

[
  {
    "step": 1,
    "question": "Qual a categoria do produto?",
    "input_used": {
      "product_category": "industrial"
    },
    "result": "industrial",
    "reason": "Categoria informada externamente (não inferida)."
  },
  {
    "step": 2,
    "question": "A Declaração Juramentada de Origem contém as informações obrigatórias (validação existente)?",
    "input_used": {
      "overall_status": "MANUAL_VERIFICATION"
    },
    "result": false,
    "reason": "Status geral da validação existente: MANUAL_VERIFICATION."
  }
]

(Summary unavailable: Gemini request failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}})

O resultado da determinação da norma de origem do ACE 18 indica que é necessária uma verificação manual da Declaração Juramentada de Origem (DJO).

No perc

In [20]:
# A couple of other branches, for a quick sanity check without re-running
# the whole pipeline each time (reuses the validation_result already
# computed above to skip the Gemini-backed base validation call):

automotive_result = evaluate_origin_rule(EXAMPLE_JSON_DJO, product_category="automotive")
print("automotive ->", automotive_result["final_status"])

game_pending_result = evaluate_origin_rule(
    EXAMPLE_JSON_DJO, product_category="game", article14_compliant=None,
    validation_result=full_report["validation_result"],
)
print("game, Article 14 answer not yet provided ->", game_pending_result["final_status"],
      "|", game_pending_result["decision_trace"][-1]["reason"])

automotive -> NOT_APPLICABLE
game, Article 14 answer not yet provided -> MANUAL_VERIFICATION | Status geral da validação existente: MANUAL_VERIFICATION.


In [21]:
# Optional presentation step — requires GEMINI_API_KEY in .env. If it is
# not configured, this simply prints an explanatory placeholder instead of
# failing the whole notebook (see generate_summary's fallback above).
summary_text = generate_summary(validation_result)
print(summary_text)


(Summary unavailable: Gemini request failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}})


In [22]:
EXAMPLE_JSON_DJO_1 = {
    "Código da aprovação DJO": "",
    "Número da DJO": "5645787",
    "Data de apresentação": "",
    "Razão social do produtor": {
        "Razão social": "INCASA S/A",
        "CNPJ/CPF": "84.689.090/0002-40",
        "Inscrição Estadual": ""
    },
    "Domicílio legal e parque industrial do produtor": {
        "Endereço": "ESTRADA DONA FRANCISCA, 11700, 89239-270 PIRABEIRABA -, JOINVILLE -SC",
        "Tel": "(47) 3205-7000",
        "E-mail": "camily.stolf@incasa.ind.br"
    },
    "Razão social do exportador": {
        "Razão social": "Incasa S/A",
        "CNPJ/CPF": "84.689.090/0001-60",
        "Inscrição Estadual": "250781360"
    },
    "Domicílio legal e parque industrial do exportador": {
        "Endereço": "Rua Dona Francisca, 11.700 - Bairro PIRABEIRABA - CEP 89239-270 - JOINVILLE - SC - BRASIL",
        "Tel": "4732057000",
        "E-mail": "camila.kricheldorf@incasa.ind.br"
    },
    "Código NCM": "2827.60.12",
    "NALADI": "2827.60.20",
    "Denominação comercial do produto a exportar": "Cloretos, oxicloretos e hidroxicloretos; brometos e oxibrometos; iodetos e oxiiodetos. lodetos e oxiiodetos. lodetos. De potássio. 528380 - IODETO DE POTÁSSIO 430363 - IODETO DE POTÁSSIO ACS 400911 - IODETO DE POTÁSSIO ESTABILIZADO 528398 - IODETO DE POTÁSSIO FLAKES 528380 - POTASSIUM IODIDE",
    "Valor FOB (USD)": "30,00",
    "Unidade de medida": "KG",
    "Descrição do processo produtivo": "O lodo é reagido com Hidróxido de Potássio onde obtêm-se 80% de lodeto de Potássio e 20% de lodato de Potássio. Em seguida é reagido com ácido fórmico para transformação do lodato de Potássio para lodeto de Potássio. Após esta etapa o lodeto de Potássio é filtrado para retirada das impurezas insolúveis e a solução filtrada é evaporada para obtenção dos cristais de lodeto de Potássio. Os sólidos são separados mediante centrifugação e os cristais são lavados para a retirada de impurezas solúveis. A última etapa é a da secagem onde é utilizado estufas de bandeja à 90℃. Consequentemente, o produto sofre alteração da sua identidade molecular.",
    "Materiales": {
        "originarios_estado_parte_produtor": {
            "Items": [
                {
                    "NCM/SH": "2815.20.00",
                    "Descrição": "Hidróxido de sódio (soda cáustica); hidróxido de potássio (potassa cáustica); peróxidos de sódio ou de potássio. Hidróxido de potássio (potassa cáustica).",
                    "País origem": "BRASIL",
                    "Valor (US$)": "4,85",
                    "% s/Valor FOB": "16,17%",
                    "Fornecedor/Fabricante": "Indefinido"
                }
            ],
            "Somatório": "16,17%"
        },
        "originarios_outros_estados_partes": {
            "Items": [
                {
                    "NCM/SH": "2801.20.90",
                    "Descrição": "Flúor, cloro, bromo e iodo. lodo. Outros. CHILE",
                    "País origem": "",
                    "Valor (US$)": "10,15",
                    "% s/Valor FOB": "33,83%",
                    "Fornecedor/Fabricante": "Indefinido"
                }
            ],
            "Somatório": "33,83%"
        },
        "nao_originarios": {
            "Items": [],
            "Somatório": ""
        },
        "terceiros_paises_ptc": {
            "Items": [],
            "Somatório": ""
        },
        "Porcentagem Total de Matérias Primas, Componentes ou Partes": "50%",
        "Valor agregado no processo Industrial (Deduzidos os tributos restituídos ou a restituir em caso de exportação)": "50%",
        "Preço FOB": "100%"
    },
    "Observações": ""
}
validation_result_1 = validate_djo(EXAMPLE_JSON_DJO_1, agreement="ACE_18")
print(json.dumps(validation_result_1, indent=2, ensure_ascii=False))

{
  "overall_status": "FAIL",
  "agreement": "ACE_18",
  "validations": [
    {
      "rule": "djo_approval",
      "status": "NOT_APPLICABLE",
      "message": "The DJO does not have both an approval code and a presentation date yet; it has not been approved."
    },
    {
      "rule": "producer_information",
      "status": "PASS",
      "message": "Producer identity and domicile are present."
    },
    {
      "rule": "ncm_ace18",
      "status": "PASS",
      "message": "NCM '2827.60.12' matched ACE_18 rule: MSP ou MaxMNO 45% ou Reação química",
      "mercosul_rule": {
        "matched_ncm_expression": "ex Capítulo 28",
        "raw_rule": "MSP ou MaxMNO 45% ou Reação química"
      }
    },
    {
      "rule": "mandatory_fields",
      "status": "PASS",
      "message": "All mandatory fields are present."
    },
    {
      "rule": "process_materials",
      "status": "FAIL",
      "message": "Some process inputs were not found in the material tables: ácido fórmico.",
      "de

In [23]:
summary_text = generate_summary(validation_result_1)
print(summary_text)


(Summary unavailable: Gemini request failed: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.6-flash\nPlease retry in 47.910580918s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'glob

## Future extension points (not implemented here, by design)

- **Decision flow**: a business-decision step consuming `validation_result`
  (e.g. auto-approve / route to manual review / reject) belongs *after*
  this notebook's output, as its own stage — `validate_djo`'s return shape
  is stable and self-contained enough to be that stage's only input.
- **More agreements**: register a new entry in `AGREEMENT_SERVICES`
  (Rule 3) with a service exposing the same `.query_ncm(ncm) -> {"found",
  "match_info", "mercosul_rule"}` shape as `ACE18Service` — no other rule
  needs to change.
- **Migrating to `backend/`**: every function above is already
  self-contained and free of notebook-only state, split along the lines of
  `validation/rules.py` (Rules 1-2, 4, 8), `validation/ncm_validator.py`
  (Rule 3 + `ace18_service.py`), `validation/table_validator.py` (Rules 6-7),
  `validation/process_validator.py` (Rule 5 + `gemini_service.py`), and
  `validation/djo_validator.py` (`validate_djo` orchestration). `models/`
  would formalize `ValidationStatus` and the result dict shapes as real
  types (dataclasses/Pydantic) instead of plain dicts.
